# Uganda Climate Policy Chatbot — XGBoost Verification

This notebook proves that the chatbot runs a real XGBoost surrogate model.  
We replay the exact lever values from a logged interaction and confirm the surrogate produces identical outputs.

**Log entry used**: entry 33 — *"What happens if there is a strong policy for clean cooking, an increase in reforestation, and better practices in agriculture?"*

In [12]:
import sys
import json
import pandas as pd
from pathlib import Path

# Add backend and its parent so 'from backend.config import ...' resolves
BACKEND_DIR = (Path('..') / 'chatbot' / 'backend').resolve()
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(BACKEND_DIR.parent))

from services.predictor import SurrogatePredictor, SectorPredictor

print('SurrogatePredictor and SectorPredictor imported successfully')

SurrogatePredictor and SectorPredictor imported successfully


## Step 1 — What the chatbot logged

Below is the exact content of **log entry 33** from `interactions.jsonl`:  
the user question, the tool call the agent made, and the simulation outputs returned.

In [13]:
LOG_PATH = Path('..') / 'chatbot_deploy' / 'logs' / 'interactions.jsonl'

with open(LOG_PATH) as f:
    entries = [json.loads(line) for line in f]

entry = entries[33]

print('=== USER QUESTION ===')
print(entry['user_message'])
print()
print('=== AGENT TOOL CALL ===')
tc = entry['tool_calls'][0]
print(f"Tool: {tc['tool']}")
print(f"Scenario name: {tc['inputs']['scenario_name']}")
print(f"Lever overrides: {tc['inputs']['lever_overrides']}")
print()
print(f'Timestamp : {entry["ts"]}')
print(f'Latency   : {entry["latency_s"]}s')

=== USER QUESTION ===
What happens if there is a strong policy for clean cooking, an increase in reforestation, and better practices in agriculture?

=== AGENT TOOL CALL ===
Tool: run_simulation
Scenario name: Clean Cooking + Reforestation + Better Agriculture
Lever overrides: {'34': 0.9, '33': 0.9, '31': 0.9, '21': 0.9, '19': 0.9, '3': 0.9, '4': 0.9, '36': 0.9, '2': 0.9, '1': 0.9}

Timestamp : 2026-05-25T22:22:38.953597+00:00
Latency   : 21.07s


In [14]:
# Lever table — what each group_id means
lever_labels = {
    34: ('Building Fuel Mix Shift to Clean Fuels', 'Buildings'),
    33: ('Biomass Cooking Stove Efficiency',       'Buildings'),
    31: ('Building Heat Energy Demand Reduction',  'Buildings'),
    21: ('Reforestation (Active Forest Expansion)','Land Use'),
    19: ('Deforestation Reduction',                'Land Use'),
     3: ('Conservation Agriculture (No-Till)',     'Agriculture'),
     4: ('Crop Yield Improvement',                 'Agriculture'),
    36: ('Nitrogen Fertilizer Reduction',          'Agriculture'),
     2: ('Agricultural Food Loss Reduction',       'Agriculture'),
     1: ('Rice Paddy Methane Reduction',           'Agriculture'),
}

log_levers = {int(k): v for k, v in tc['inputs']['lever_overrides'].items()}

lever_rows = []
for gid, (name, sector) in lever_labels.items():
    lever_rows.append({
        'Group ID': gid,
        'Policy Lever': name,
        'Sector': sector,
        'BAU (L)': 0.1,
        'Scenario (L)': log_levers.get(gid, 0.1),
    })

df_levers = pd.DataFrame(lever_rows)
print('Levers activated by the chatbot agent:')
df_levers

Levers activated by the chatbot agent:


,Group ID,Policy Lever,Sector,BAU (L),Scenario (L)
0,34,Building Fuel Mix Shift to Clean Fuels,Buildings,0.1,0.9
1,33,Biomass Cooking Stove Efficiency,Buildings,0.1,0.9
2,31,Building Heat Energy Demand Reduction,Buildings,0.1,0.9
3,21,Reforestation (Active Forest Expansion),Land Use,0.1,0.9
4,19,Deforestation Reduction,Land Use,0.1,0.9
5,3,Conservation Agriculture (No-Till),Agriculture,0.1,0.9
6,4,Crop Yield Improvement,Agriculture,0.1,0.9
7,36,Nitrogen Fertilizer Reduction,Agriculture,0.1,0.9
8,2,Agricultural Food Loss Reduction,Agriculture,0.1,0.9
9,1,Rice Paddy Methane Reduction,Agriculture,0.1,0.9


In [15]:
# Logged simulation outputs (subset of key metrics for comparison)
log_outputs = entry['simulation_outputs']

KEY_METRICS = [
    ('emission_total_sum',           'Total Emissions 2025–2070',     'Mt CO₂e'),
    ('2033_2037_mean_emission',       'Near-Term Emissions (2033–37)', 'Mt CO₂e/yr'),
    ('2066_2070_mean_emission',       'Long-Term Emissions (2066–70)', 'Mt CO₂e/yr'),
    ('2025_2070_mean_benefits',       'Long-Term Co-Benefits (avg)',   'B USD/yr'),
    ('2025_2070_mean_costs',          'Long-Term Costs (avg)',         'B USD/yr'),
    ('2025_2070_max_costs_rel_to_gdp','Peak Cost as % of GDP',        '% GDP'),
]

SECTOR_METRICS_2070 = [
    ('emission_scoe_yr2070', 'SCOE 2070 (Buildings/Cooking)', 'Mt CO₂e/yr'),
    ('emission_frst_yr2070', 'FRST 2070 (Forestry)',          'Mt CO₂e/yr'),
    ('emission_lndu_yr2070', 'LNDU 2070 (Land Use)',          'Mt CO₂e/yr'),
    ('emission_agrc_yr2070', 'AGRC 2070 (Agriculture)',       'Mt CO₂e/yr'),
    ('emission_lvst_yr2070', 'LVST 2070 (Livestock)',         'Mt CO₂e/yr'),
]

rows = []
for key, label, unit in KEY_METRICS + SECTOR_METRICS_2070:
    val = log_outputs.get(key)
    display_val = f'{val * 100:.2f}%' if 'gdp' in key and val is not None else (f'{val:.4f}' if val is not None else 'N/A')
    rows.append({'Metric': label, 'Unit': unit, 'Chatbot Log Value': display_val, '_raw': val, '_key': key})

df_log = pd.DataFrame(rows)
print('Chatbot log outputs (entry 33):')
df_log[['Metric', 'Unit', 'Chatbot Log Value']]

Chatbot log outputs (entry 33):


,Metric,Unit,Chatbot Log Value
0,Total Emissions 2025–2070,Mt CO₂e,6978.6050
1,Near-Term Emissions (2033–37),Mt CO₂e/yr,138.4959
2,Long-Term Emissions (2066–70),Mt CO₂e/yr,149.6741
3,Long-Term Co-Benefits (avg),B USD/yr,17.0323
4,Long-Term Costs (avg),B USD/yr,0.6715
5,Peak Cost as % of GDP,% GDP,0.51%
6,SCOE 2070 (Buildings/Cooking),Mt CO₂e/yr,6.1450
7,FRST 2070 (Forestry),Mt CO₂e/yr,-17.5130
8,LNDU 2070 (Land Use),Mt CO₂e/yr,39.5020
9,AGRC 2070 (Agriculture),Mt CO₂e/yr,4.1060


## Step 2 — Run XGBoost directly with the same levers

We now load the trained XGBoost pipeline and call `predict_comparison()` with the exact same lever values.  
If the values match the log, it proves the chatbot is a genuine surrogate model interface.

In [16]:
predictor = SurrogatePredictor()
print(f'Model loaded. Feature count: {len(predictor._feature_columns)}')

Model loaded. Feature count: 68


In [17]:
# Run with the same lever overrides from the log
LEVER_OVERRIDES = {34: 0.9, 33: 0.9, 31: 0.9, 21: 0.9, 19: 0.9, 3: 0.9, 4: 0.9, 36: 0.9, 2: 0.9, 1: 0.9}

result = predictor.predict_comparison(
    lever_overrides=LEVER_OVERRIDES,
    scenario_name='Clean Cooking + Reforestation + Better Agriculture',
)

scenario_preds = result['scenario']['predictions']
baseline_preds = result['baseline']['predictions']

sector_predictor = SectorPredictor()
sector_result = sector_predictor.predict_comparison(
    lever_overrides=LEVER_OVERRIDES,
    scenario_name='Clean Cooking + Reforestation + Better Agriculture',
)

sector_scenario = sector_result['scenario']['sector_trajectories']
sector_baseline = sector_result['baseline']['sector_trajectories']

print('XGBoost run complete.')
print(f"Scenario: {result['scenario']['scenario_name']}")
print(f"Baseline: {result['baseline']['scenario_name']}")

XGBoost run complete.
Scenario: Clean Cooking + Reforestation + Better Agriculture
Baseline: Business as Usual (Baseline)


## Step 3 — Side-by-side comparison: Log vs Direct XGBoost

Every row should show **Match ✓** — confirming the chatbot routes directly through the surrogate without modification.

In [18]:
comparison_rows = []

for key, label, unit in KEY_METRICS:
    log_val = log_outputs.get(key)
    xgb_val = scenario_preds[key]['value'] if key in scenario_preds else None
    if 'gdp' in key:
        log_disp = f'{log_val * 100:.3f}%' if log_val is not None else 'N/A'
        xgb_disp = f'{xgb_val * 100:.3f}%' if xgb_val is not None else 'N/A'
    else:
        log_disp = f'{log_val:.4f}' if log_val is not None else 'N/A'
        xgb_disp = f'{xgb_val:.4f}' if xgb_val is not None else 'N/A'
    match = '✓' if (log_val is not None and xgb_val is not None and abs(log_val - xgb_val) < 0.001) else '✗'
    comparison_rows.append({'Metric': label, 'Unit': unit, 'Chatbot Log': log_disp, 'XGBoost Direct': xgb_disp, 'Match': match})

for key, label, unit in SECTOR_METRICS_2070:
    parts = key.replace('emission_', '').split('_yr')
    sector_code, year = parts[0], int(parts[1])
    log_val = log_outputs.get(key)
    xgb_val = sector_scenario.get(sector_code, {}).get(year)
    log_disp = f'{log_val:.4f}' if log_val is not None else 'N/A'
    xgb_disp = f'{xgb_val:.4f}' if xgb_val is not None else 'N/A'
    match = '✓' if (log_val is not None and xgb_val is not None and abs(log_val - xgb_val) < 0.001) else '✗'
    comparison_rows.append({'Metric': label, 'Unit': unit, 'Chatbot Log': log_disp, 'XGBoost Direct': xgb_disp, 'Match': match})

df_comparison = pd.DataFrame(comparison_rows)
print('Comparison — chatbot log vs direct XGBoost inference:')
df_comparison

Comparison — chatbot log vs direct XGBoost inference:


,Metric,Unit,Chatbot Log,XGBoost Direct,Match
0,Total Emissions 2025–2070,Mt CO₂e,6978.6050,6978.6050,✓
1,Near-Term Emissions (2033–37),Mt CO₂e/yr,138.4959,138.4959,✓
2,Long-Term Emissions (2066–70),Mt CO₂e/yr,149.6741,149.6741,✓
3,Long-Term Co-Benefits (avg),B USD/yr,17.0323,17.0323,✓
4,Long-Term Costs (avg),B USD/yr,0.6715,0.6715,✓
5,Peak Cost as % of GDP,% GDP,0.510%,0.510%,✓
6,SCOE 2070 (Buildings/Cooking),Mt CO₂e/yr,6.1450,6.1450,✓
7,FRST 2070 (Forestry),Mt CO₂e/yr,-17.5130,-17.5130,✓
8,LNDU 2070 (Land Use),Mt CO₂e/yr,39.5020,39.5020,✓
9,AGRC 2070 (Agriculture),Mt CO₂e/yr,4.1060,4.1060,✓


In [19]:
from services.predictor import SectorPredictor

sector_predictor = SectorPredictor()
sector_result = sector_predictor.predict_comparison(
    lever_overrides=LEVER_OVERRIDES,
    scenario_name='Clean Cooking + Reforestation + Better Agriculture',
)

sector_scenario = sector_result['scenario']['sector_trajectories']
sector_baseline = sector_result['baseline']['sector_trajectories']

sector_rows = []
for key, label, unit in SECTOR_METRICS_2070:
    # Parse "emission_scoe_yr2070" → sector="scoe", year=2070
    parts = key.replace('emission_', '').split('_yr')
    sector_code, year = parts[0], int(parts[1])

    log_val = log_outputs.get(key)
    xgb_val = sector_scenario.get(sector_code, {}).get(year)
    bau_val = sector_baseline.get(sector_code, {}).get(year)

    log_disp = f'{log_val:.4f}' if log_val is not None else 'N/A'
    xgb_disp = f'{xgb_val:.4f}' if xgb_val is not None else 'N/A'
    bau_disp = f'{bau_val:.4f}' if bau_val is not None else 'N/A'

    match = '✓' if (log_val is not None and xgb_val is not None and abs(log_val - xgb_val) < 0.001) else '✗'

    pct = None
    if xgb_val is not None and bau_val and bau_val != 0:
        pct = 100 * (xgb_val - bau_val) / abs(bau_val)
    pct_disp = f'{"▼" if pct < 0 else "▲"} {abs(pct):.1f}%' if pct is not None else 'N/A'

    sector_rows.append({
        'Metric': label,
        'Unit': unit,
        'Chatbot Log': log_disp,
        'Sector XGBoost': xgb_disp,
        'BAU Baseline': bau_disp,
        'Match': match,
        'vs BAU': pct_disp,
    })

df_sector = pd.DataFrame(sector_rows)
print('Sector comparison — chatbot log vs direct SectorPredictor inference:')
df_sector

Sector comparison — chatbot log vs direct SectorPredictor inference:


,Metric,Unit,Chatbot Log,Sector XGBoost,BAU Baseline,Match,vs BAU
0,SCOE 2070 (Buildings/Cooking),Mt CO₂e/yr,6.1450,6.1450,79.4250,✓,▼ 92.3%
1,FRST 2070 (Forestry),Mt CO₂e/yr,-17.5130,-17.5130,-11.0940,✓,▼ 57.9%
2,LNDU 2070 (Land Use),Mt CO₂e/yr,39.5020,39.5020,68.3140,✓,▼ 42.2%
3,AGRC 2070 (Agriculture),Mt CO₂e/yr,4.1060,4.1060,4.7510,✓,▼ 13.6%
4,LVST 2070 (Livestock),Mt CO₂e/yr,63.5580,63.5580,57.6110,✓,▲ 10.3%


In [20]:
# Chatbot reply snippet from the log
print('=== CHATBOT REPLY (from log) ===')
print(entry['reply_snippet'])

=== CHATBOT REPLY (from log) ===
This three-pillar package delivers **dramatic long-term reductions in cooking/buildings and land use**, though national totals are partially offset by livestock growth. Here are the results:

## Emissions Impact: Clean Cooking + Reforestation + Better Agriculture

| Metric | BAU | Scenario | Change 


In [21]:
# Save results for the slides script
output = {
    'log_entry': {
        'user_message': entry['user_message'],
        'ts': entry['ts'],
        'latency_s': entry['latency_s'],
        'lever_overrides': log_levers,
        'simulation_outputs': log_outputs,
        'reply_snippet': entry['reply_snippet'],
    },
    'xgboost_direct': {
        'scenario': {k: v['value'] for k, v in scenario_preds.items()},
        'baseline': {k: v['value'] for k, v in baseline_preds.items()},
        'comparison_pct': result['comparison'],
    },
    'lever_labels': {str(k): {'name': v[0], 'sector': v[1]} for k, v in lever_labels.items()},
}

out_path = Path('demo_comparison_results.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Results saved to {out_path.resolve()}')

Results saved to /Users/alexa/Projects/ssp_uganda_data/metamodel/scripts/demo_comparison_results.json
